# Curse of Dimensionality bei KNN

Mit jeder zusätzlichen Dimension wächst der Raum stark. Daten werden dünn verteilt und der nächste Nachbar ist relativ kaum noch näher als viele andere Punkte. Wir fügen absichtlich irrelevante Merkmale hinzu.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.feature_selection import SelectKBest,f_classif
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import cross_val_score,train_test_split,GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng=np.random.default_rng(42)
X_basis,y=make_moons(n_samples=800,noise=.24,random_state=42)
dimensionen=[2,4,7,12,22,52,102]
result=[]
for d in dimensionen:
 rauschen=rng.normal(size=(len(X_basis),d-2))
 X=np.c_[X_basis,rauschen]
 modell=Pipeline([("scale",StandardScaler()),("knn",KNeighborsClassifier(n_neighbors=9))])
 scores=cross_val_score(modell,X,y,cv=5,scoring="f1")
 # Distanzkonzentration für eine standardisierte Stichprobe
 Z=StandardScaler().fit_transform(X[:250]); D=pairwise_distances(Z)
 np.fill_diagonal(D,np.nan)
 nahe=np.nanmin(D,axis=1); ferne=np.nanmax(D,axis=1)
 result.append({"Dimensionen":d,"CV-F1":scores.mean(),"F1 Std":scores.std(),"nah/fern":np.mean(nahe/ferne)})
result=pd.DataFrame(result); display(result.round(3))
fig,ax=plt.subplots(1,2,figsize=(12,4)); result.plot(x="Dimensionen",y="CV-F1",marker="o",ax=ax[0]); result.plot(x="Dimensionen",y="nah/fern",marker="o",color="tomato",ax=ax[1]); ax[0].set_ylim(.45,1); ax[1].set_ylim(0,1); ax[1].set_ylabel("mittlere Distanz nächster / fernster"); plt.show()

## Was passiert?

- Die zusätzlichen Merkmale enthalten keine Zielinformation, zählen aber trotzdem zur Distanz.
- Das Verhältnis „nächster zu fernster Abstand“ nähert sich 1: Nachbarschaften werden weniger eindeutig.
- Mehr Merkmale bedeuten nicht automatisch mehr Information.

Feature Selection muss innerhalb der Pipeline und jedes CV-Folds gelernt werden, sonst entsteht Data Leakage.

In [ ]:
X_viele=np.c_[X_basis,rng.normal(size=(len(X_basis),100))]
X_train,X_test,y_train,y_test=train_test_split(X_viele,y,test_size=.25,random_state=42,stratify=y)
auswahl_pipeline=Pipeline([
 ("scale",StandardScaler()),
 ("auswahl",SelectKBest(score_func=f_classif)),
 ("knn",KNeighborsClassifier())])
grid={"auswahl__k":[2,5,10,25,"all"],"knn__n_neighbors":[5,9,15],"knn__weights":["uniform","distance"]}
suche=GridSearchCV(auswahl_pipeline,grid,scoring="f1",cv=5,n_jobs=-1).fit(X_train,y_train)
print("Beste Parameter:",suche.best_params_)
print("Beste CV-F1:",round(suche.best_score_,3),"Test-F1/Score:",round(suche.score(X_test,y_test),3))

vergleich=[]
for k in [2,5,10,25,"all"]:
 m=Pipeline([("scale",StandardScaler()),("auswahl",SelectKBest(f_classif,k=k)),("knn",KNeighborsClassifier(n_neighbors=9))])
 vergleich.append({"Merkmale":str(k),"CV-F1":cross_val_score(m,X_viele,y,cv=5,scoring="f1").mean()})
pd.DataFrame(vergleich).plot.bar(x="Merkmale",y="CV-F1",ylim=(.45,1),legend=False,color="mediumpurple"); plt.ylabel("CV-F1"); plt.show()

## Einordnung

Weniger Merkmale helfen nur, wenn überwiegend irrelevante oder redundante Dimensionen entfernt werden. Auswahl ist selbst ein Hyperparameter und muss validiert werden. Alternativen sind Fachwissen, Regularisierung, PCA oder mehr Daten. Bei extrem hohen Dimensionen sind andere Modellfamilien oft geeigneter.